In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import random
 

In [0]:
random.seed(42)
 
# Listening events (large table - 500K rows)
events_data = []
for i in range(500000):
    events_data.append((
        f"EVT-{i+1:07d}",
        f"USR-{random.randint(1, 100000):06d}",
        f"TRK-{random.randint(1, 50000):06d}",
        f"ART-{random.randint(1, 5000):05d}",
        random.randint(10, 300),
        random.choice([True, False]),
        random.choice(["mobile", "desktop", "smart_speaker", "tablet"]),
        random.choice(["free", "premium"]),
        f"202{random.randint(3,4)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
    ))
 
events = spark.createDataFrame(events_data,
    ["event_id", "user_id", "track_id", "artist_id", "duration_sec",
     "completed", "device", "tier", "event_date"]) \
    .withColumn("event_date", col("event_date").cast("date")) \
    .withColumn("year", year(col("event_date"))) \
    .withColumn("month", month(col("event_date")))
 
events.write.parquet("/Volumes/workspace/default/demo/events", mode="overwrite", partitionBy=["year"])
 
# Artists (small table - 5K rows)
artist_data = [(f"ART-{i+1:05d}", f"Artist {i+1}",
                random.choice(["Pop", "Rock", "Hip-Hop", "Jazz", "Electronic"]),
                random.choice(["US", "UK", "KR", "JP", "DE"]))
               for i in range(5000)]
artists = spark.createDataFrame(artist_data, ["artist_id", "name", "genre", "country"])
artists.write.parquet("/Volumes/workspace/default/demo/artists", mode="overwrite")
 
# Tracks (medium table - 50K rows)
track_data = [(f"TRK-{i+1:06d}", f"Track {i+1}",
               f"ART-{random.randint(1, 5000):05d}",
               random.randint(60, 400),
               random.randint(2018, 2024))
              for i in range(50000)]
tracks = spark.createDataFrame(track_data,
    ["track_id", "title", "artist_id", "track_duration", "release_year"])
tracks.write.parquet("/Volumes/workspace/default/demo/tracks", mode="overwrite")
 
# Reload from Parquet
events = spark.read.parquet("/Volumes/workspace/default/demo/events")
artists = spark.read.parquet("/Volumes/workspace/default/demo/artists")
tracks = spark.read.parquet("/Volumes/workspace/default/demo/tracks")
 
print(f"Events: {events.count()} | Artists: {artists.count()} | Tracks: {tracks.count()}")

Events: 500000 | Artists: 5000 | Tracks: 50000


In [0]:
q1 = events.filter(col("year") == 2024) \
    .filter(col("completed") == True) \
    .select("event_id", "user_id", "duration_sec")
 
print("QUERY 1: Simple filter and select")
q1.explain(mode="formatted")

QUERY 1: Simple filter and select
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [5]: [event_id#28281, user_id#28282, duration_sec#28285L, completed#28286, year#28291]
DictionaryFilters: [completed#28286]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/demo/events]
PartitionFilters: [isnotnull(year#28291), (year#28291 = 2024)]
ReadSchema: struct<event_id:string,user_id:string,duration_sec:bigint,completed:boolean>
RequiredDataFilters: [completed#28286, isnotnull(completed#28286)]

(2) PhotonProject
Input [5]: [event_id#28281, user_id#28282, duration_sec#28285L, completed#28286, year#28291]
Arguments: [event_id#28281, user_id#28282, duration_sec#28285L]

(3) PhotonColumnarToRow
Input [3]: [event_id#28281, user_id#28282, duration_sec#28285L]

(4) PhotonResultStage
Input [3]: [event_id#28281, user_id#28282, duration_sec#28285L]


== Photon Explanation ==
The query i

In [0]:

# ect	Your Finding
# Scan type: PhotonScan parquet  
# PartitionFilters: [isnotnull(year#28291), (year#28291 = 2024)]
# PushedFilters	___
# ReadSchema columns: struct<event_id:string,user_id:string,duration_sec:bigint,completed:boolean>
# Exchange count	___
# Assessment	___

In [0]:
q2 = events.join(artists, "artist_id") \
    .filter(col("year") == 2024) \
    .select("event_id", "name", "genre", "duration_sec")
 
print("QUERY 2: Events JOIN Artists (filter after join)")
q2.explain(mode="formatted")

QUERY 2: Events JOIN Artists (filter after join)
== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonProject (8)
         +- PhotonBroadcastHashJoin Inner (7)
            :- PhotonProject (2)
            :  +- PhotonScan parquet  (1)
            +- PhotonShuffleExchangeSource (6)
               +- PhotonShuffleMapStage (5)
                  +- PhotonShuffleExchangeSink (4)
                     +- PhotonScan parquet  (3)


(1) PhotonScan parquet 
Output [4]: [event_id#28281, artist_id#28284, duration_sec#28285L, year#28291]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/demo/events]
OptionalDataFilters: [hashedrelationcontains(artist_id#28284)]
PartitionFilters: [isnotnull(year#28291), (year#28291 = 2024)]
ReadSchema: struct<event_id:string,artist_id:string,duration_sec:bigint>
RequiredDataFilters: [isnotnull(artist_id#28284)]

(2) PhotonProject
Input [4]: [event_id#28281, artist_id#28284, 

In [0]:

# Aspect	Your Finding
# Join strategy	Inner
# Artists table size	~5K rows (small!)
# Exchange count	___
# Could broadcast?	___
# Filter placement	after join
# Assessment	___

In [0]:
q3 = events.join(tracks, "track_id") \
    .join(artists, "artist_id") \
    .filter(col("year") == 2024) \
    .filter(col("genre") == "Pop") \
    .groupBy("name") \
    .agg(count("*").alias("play_count"), avg("duration_sec").alias("avg_duration"))
 
print("QUERY 3: Three-table join with aggregation")
q3.explain(mode="formatted")

QUERY 3: Three-table join with aggregation
== Physical Plan ==
AdaptiveSparkPlan (23)
+- == Initial Plan ==
   PhotonResultStage (22)
   +- PhotonColumnarToRow (21)
      +- PhotonGroupingAgg (20)
         +- PhotonShuffleExchangeSource (19)
            +- PhotonShuffleMapStage (18)
               +- PhotonShuffleExchangeSink (17)
                  +- PhotonGroupingAgg (16)
                     +- PhotonProject (15)
                        +- PhotonBroadcastHashJoin Inner (14)
                           :- PhotonProject (8)
                           :  +- PhotonBroadcastHashJoin Inner (7)
                           :     :- PhotonProject (2)
                           :     :  +- PhotonScan parquet  (1)
                           :     +- PhotonShuffleExchangeSource (6)
                           :        +- PhotonShuffleMapStage (5)
                           :           +- PhotonShuffleExchangeSink (4)
                           :              +- PhotonScan parquet  (3)
            

In [0]:

# Aspect	Your Finding
# Join 1 strategy	inner
# Join 2 strategy	inner
# Total Exchange count	___
# Filter on year?	___ (pushed to partition?)
# Filter on genre?	___ (pushed to scan?)
# Assessment	___

In [0]:
enriched = events.join(artists, "artist_id").filter(col("year") == 2024)
 
print("QUERY 4a: Genre aggregation")
q4a = enriched.groupBy("genre").agg(count("*").alias("plays"))
q4a.explain(mode="formatted")
 
print("\nQUERY 4b: Device aggregation (same enriched source)")
q4b = enriched.groupBy("device").agg(avg("duration_sec").alias("avg_dur"))
q4b.explain(mode="formatted")

QUERY 4a: Genre aggregation
== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonProject (8)
                        +- PhotonBroadcastHashJoin Inner (7)
                           :- PhotonProject (2)
                           :  +- PhotonScan parquet  (1)
                           +- PhotonShuffleExchangeSource (6)
                              +- PhotonShuffleMapStage (5)
                                 +- PhotonShuffleExchangeSink (4)
                                    +- PhotonScan parquet  (3)


(1) PhotonScan parquet 
Output [2]: [artist_id#28284, year#28291]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/demo/events]
OptionalDataFilters: [hashedrelationcon

In [0]:

# Aspect	Your Finding
# Does 4a and 4b share computation?	no
# Is enriched cached?	yes
# Redundant work	no
# Assessment	___

In [0]:
popular = events.groupBy("track_id").agg(count("*").alias("play_count")) \
    .filter(col("play_count") > 10)
 
q5 = events.join(popular, "track_id") \
    .select("event_id", "user_id", "track_id", "play_count")
 
print("QUERY 5: Self-reference (events aggregated then joined back)")
q5.explain(mode="formatted")

QUERY 5: Self-reference (events aggregated then joined back)
== Physical Plan ==
AdaptiveSparkPlan (18)
+- == Initial Plan ==
   PhotonResultStage (17)
   +- PhotonColumnarToRow (16)
      +- PhotonProject (15)
         +- PhotonBroadcastHashJoin Inner (14)
            :- PhotonProject (2)
            :  +- PhotonScan parquet  (1)
            +- PhotonShuffleExchangeSource (13)
               +- PhotonShuffleMapStage (12)
                  +- PhotonShuffleExchangeSink (11)
                     +- PhotonFilter (10)
                        +- PhotonGroupingAgg (9)
                           +- PhotonShuffleExchangeSource (8)
                              +- PhotonShuffleMapStage (7)
                                 +- PhotonShuffleExchangeSink (6)
                                    +- PhotonGroupingAgg (5)
                                       +- PhotonProject (4)
                                          +- PhotonScan parquet  (3)


(1) PhotonScan parquet 
Output [4]: [event_id#28281,

In [0]:

# Aspect	Your Finding
# How many times is events scanned?	3
# Exchange count	___
# Join strategy	inner
# Could caching help?	___
# Assessment	___